In [ ]:
#importing all the required modules
import numpy as np # linear algebra
import pandas as pd
pd.set_option("display.max_rows", 101)
import os
import cv2
import json
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["font.size"] = 15
import seaborn as sns
from collections import Counter
from PIL import Image
import math
import seaborn as sns

In [ ]:
!mv kaggle.json /root/.kaggle/
!kaggle competitions download -c imaterialist-fashion-2019-FGVC6

In [ ]:
!unzip /content/imaterialist-fashion-2019-FGVC6.zip

# Data Visulalization

In [ ]:
def classid2label(class_id):
    # Split the class_id string using the underscore "_" as the separator
    # The first part before the underscore represents the category of the label
    #the first part is catogory second part is attribute
    category, *attribute = class_id.split("_")
    return category, attribute

In [ ]:
def print_dict(dictionary, name_dict):
     # Printing column headers
    print("{}{}{}{}{}".format("rank".ljust(5), "id".center(8), "name".center(40), "amount".rjust(10), "ratio(%)".rjust(10)))
    all_num = sum(dictionary.values())
    for i, (key, val) in enumerate(sorted(dictionary.items(), key=lambda x: -x[1])):
        print("{:<5}{:^8}{:^40}{:>10}{:>10.3%}".format(i+1, key, name_dict[key], val, val/all_num))

In [ ]:
def print_img(img_name, ax):
    img_df = train_df[train_df.ImageId == img_name]
    labels = list(set(img_df["ClassId"].values))
    img = np.asarray(Image.open(input_dir + "train/" + img_name))
    label_interval = (img.shape[0] * 0.9) / len(labels)

    for num, attribute_id in enumerate(labels):
        x_pos = img.shape[1] * 1.1
        y_pos = (img.shape[0] * 0.9) / len(labels) * (num + 2) + (img.shape[0] * 0.1)
        if(num == 0):
            ax.text(x_pos, y_pos-label_interval*2, "category", fontsize=12)
            ax.text(x_pos, y_pos-label_interval, category_name_dict[attribute_id], fontsize=12)
            if(len(labels) > 1):
                ax.text(x_pos, y_pos, "attribute", fontsize=12)
        else:
            ax.text(x_pos, y_pos, attribute_name_dict[attribute_id], fontsize=12)

In [ ]:
def json2df(data):
    df = pd.DataFrame()
    for index, el in enumerate(data):
        for key, val in el.items():
            df.loc[index, key] = val
    return df

In [ ]:
#read the train csv file
train_df = pd.read_csv("/content/data/train.csv")

In [ ]:
# print train csv
train_df.head()

In [ ]:
# read label_descriptions.json file
with open("/workspace/data/label_descriptions.json") as f:
    label_description = json.load(f)

In [ ]:

print("this dataset info")
print(json.dumps(label_description["info"], indent=2))

In [ ]:
category_df = json2df(label_description["categories"])
category_df["id"] = category_df["id"].astype(int)
category_df["level"] = category_df["level"].astype(int)
attribute_df = json2df(label_description["attributes"])
attribute_df["id"] = attribute_df["id"].astype(int)
attribute_df["level"] = attribute_df["level"].astype(int)

In [ ]:
# printing the number of categories and attributes
print("{} categories {} attributes.".format(len(label_description['categories']), len(label_description['attributes'])))


In [ ]:
train_df.loc[train_df['ClassId'] == '7'].count()

In [ ]:
image_label_num_df = train_df.groupby("ImageId")["ClassId"].count()

In [ ]:
fig, ax = plt.subplots(figsize=(25, 7))
x_value = image_label_num_df.value_counts().index.values
y_value = image_label_num_df.value_counts().values
zipped = zip(x_value, y_value)
zipped = sorted(zipped)
x, y = zip(*zipped)
index = 0
x_list = []
y_list = []
for i in range(1, max(x)+1):
    if(i not in x):
        x_list.append(i)
        y_list.append(0)
    else:
        x_list.append(i)
        y_list.append(y[index])
        index += 1

for i, j in zip(x_list, y_list):
    ax.text(i-1, j, j, ha="center", va="bottom", fontsize=13)

sns.barplot(x=x_list, y=y_list, ax=ax)
ax.set_xticks(list(range(0, len(x_list), 5)))
ax.set_xticklabels(list(range(1, len(x_list), 5)))
ax.set_title("the number of labels per image")
ax.set_xlabel("the number of labels")
ax.set_ylabel("amount")

In [ ]:
#creating counters to track category and attribute
counter_category = Counter()
counter_attribute = Counter()
for class_id in train_df["ClassId"]:
    category, attribute = classid2label(class_id)
    counter_category.update([category])
    counter_attribute.update(attribute)

In [ ]:
len(counter_category)

In [ ]:
len(counter_attribute)

In [ ]:
# This extracts category names and attribute names from the "categories" and "attributes" sections
category_name_dict = {}
for i in label_description["categories"]:
    category_name_dict[str(i["id"])] = i["name"]
attribute_name_dict = {}
for i in label_description["attributes"]:
    attribute_name_dict[str(i["id"])] = i["name"]

In [ ]:
print("Category label frequency")
print_dict(counter_category, category_name_dict)

In [ ]:
print("Attribute label frequency")
print_dict(counter_attribute, attribute_name_dict)

In [ ]:
train_df.ClassId.max()

In [ ]:
image_shape_df = train_df.groupby("ImageId")[["Height", "Width"]].first()

In [ ]:
#creating histograms for height and width distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
ax1.hist(image_shape_df.Height, bins=100)
ax1.set_title("Height distribution")
ax2.hist(image_shape_df.Width, bins=100)
ax2.set_title("Width distribution");

In [ ]:
#ewading the data file
input_dir = '/workspace/data/'
img_name = image_shape_df.Height.idxmin()
height, width = image_shape_df.loc[img_name, :]
print("Minimam height image is {},\n(H, W) = ({}, {})".format(img_name, height, width))
fig, ax = plt.subplots()
print_img(img_name, ax)

In [ ]:
# Finding the image with the maximum height by getting its index

img_name = image_shape_df.Height.idxmax()
height, width = image_shape_df.loc[img_name, :]
#printing the image
print("Maximum height image is {},\n(H, W) = ({}, {})".format(img_name, height, width))
fig, ax = plt.subplots()
print_img(img_name, ax)

In [ ]:
#finding the image which has minimum width
img_name = image_shape_df.Width.idxmin()
height, width = image_shape_df.loc[img_name, :]
print("Minimum width image is {},\n(H, W) = ({}, {})".format(img_name, height, width))
fig, ax = plt.subplots()
print_img(img_name, ax)

In [ ]:
#getting the maximum width
img_name = image_shape_df.Width.idxmax()
height, width = image_shape_df.loc[img_name, :]
print("Maximum width image is {},\n(H, W) = ({}, {})".format(img_name, height, width))
fig, ax = plt.subplots()
print_img(img_name, ax)

In [ ]:
def make_mask_img(segment_df):

    seg_width =  segment_df.at[0, "Width"]
    seg_height = segment_df.at[0, "Height"]
    seg_img = np.full(seg_width*seg_height, category_num-1, dtype=np.float32)
    for encoded_pixels, class_id in zip(segment_df["EncodedPixels"].values, segment_df["ClassId"].values):
        pixel_list = list(map(int, encoded_pixels.split(" ")))
        for i in range(0, len(pixel_list), 2):
            start_index = pixel_list[i] - 1
            index_len = pixel_list[i+1] - 1
            seg_img[start_index:start_index+index_len] =int(class_id.split("_")[0])
    seg_img = seg_img.reshape((seg_height, seg_width), order='F')
    seg_img = cv2.resize(seg_img, (WIDTH, HEIGHT), interpolation=cv2.INTER_NEAREST)
    return seg_img



In [ ]:
#preparing training data by processing images and their corresponding masks
img_ind_num = train_df.groupby("ImageId")["ClassId"].count()
counter = train_df.index.values[0]
train_base = "/workspace/data/train/"
train_images = []
mask_images = []
WIDTH = 256
HEIGHT = 256
category_num = 13
for i, (img_name, ind_num) in enumerate(img_ind_num.items()):
    img = cv2.imread(train_base + img_name)
    img = cv2.resize(img, (WIDTH, HEIGHT), interpolation=cv2.INTER_AREA)

    mask_df = (train_df.loc[counter:counter+ind_num-1, :]).reset_index(drop=True)
    counter += ind_num
    mask_img = make_mask_img(mask_df)
    fig, ax = plt.subplots()
    ax.imshow(mask_img)
    break


# Data Pre-processing

In [ ]:
final_dataset = train_df.loc[train_df['ClassId'].apply(lambda x: x in ['0','1','2','3','4','5','6','7','8','9','10','11','12'])]
final_dataset = final_dataset.reset_index()
final_dataset.drop(["index","ImageId"], axis=1)

In [ ]:
final_dataset["ClassId"].unique()

In [ ]:
#generating batches of training data
def train_generator(train_df, batch_size):

    img_ind_num = train_df.groupby("ImageId")["ClassId"].count()
    counter = train_df.index.values[0]

    train_images = []
    mask_images = []

    for i, (img_name, ind_num) in enumerate(img_ind_num.items()):
        img = cv2.imread(train_base + img_name)
        img = cv2.resize(img, (WIDTH, HEIGHT), interpolation=cv2.INTER_AREA)

        mask_df = (train_df.loc[counter:counter+ind_num-1, :]).reset_index(drop=True)
        counter += ind_num
        mask_img = make_mask_img(mask_df)
        mask_images.append(mask_img)

        img = img.transpose((2, 0, 1))
        train_images.append(img)

        if((i+1) % batch_size == 0):
            yield np.array(train_images, dtype=np.float32) / 255, np.array(mask_images, dtype=np.int32)
            train_images = []
            mask_images = []

    if(len(train_images) != 0):
        yield np.array(train_images, dtype=np.float32) / 255, np.array(mask_images, dtype=np.int32)

In [ ]:
WIDTH = 256
HEIGHT = 256
category_num = 47
device = "cuda:0"

# Custom Dataloader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

import cv2

class CustomDataset(Dataset):
    def __init__(self, image_paths, train_df, train=True, transforms=None):
        super().__init__()
        self.image_paths = image_paths
        self.train_df = train_df
        self.img_ind_num = self.train_df.groupby("ImageId")["ClassId"].count()
        self.transforms = transforms

    def __getitem__(self, index):
        self.counter = self.train_df.index.values[0]
        train_images = []
        mask_images = []
        for i, (img_name, ind_num) in enumerate (self.img_ind_num.items()):
            self.counter += ind_num
            if index == i:
                image = cv2.imread(self.image_paths + img_name)
                image = cv2.resize(image, (WIDTH, HEIGHT), interpolation=cv2.INTER_AREA)
                mask_df = (train_df.loc[self.counter:self.counter+ind_num-1, :]).reset_index(drop=True)
                self.counter += ind_num
                mask_img = make_mask_img(mask_df)
                image = image.transpose((2, 0, 1))
                image = image.astype(np.float32)
                image = image/255

                return image, mask_img
    def __len__(self):
        return 35000

train_image_paths = '/content/data/train/'

transform = transforms.Compose([transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)), transforms.ToTensor()])
train_dataset = CustomDataset(train_image_paths, final_dataset, train=True, transforms=transform)


# Unet Model

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
import pytorch_lightning as pl

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchmetrics
import pandas as pd
from torchmetrics.functional import dice


def calculate_iou(prediction, target):
    value = dice(prediction, target)
    return value

#U-net MODEL ARCHITECTURE
class Unet(pl.LightningModule):
    def __init__(self,in_channels, out_channels):
        self.validation_step_outputs = []
        super(Unet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)

        self.conv7 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv8 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(2, 2)

        self.conv9 = nn.Conv2d(512, 1024, kernel_size=3, padding=1)
        self.conv10 = nn.Conv2d(1024, 1024, kernel_size=3, padding=1)

        self.upconv1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv11 = nn.Conv2d(1024, 512, kernel_size=3, padding=1)
        self.conv12 = nn.Conv2d(512, 512, kernel_size=3, padding=1)

        self.upconv2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv13 = nn.Conv2d(512, 256, kernel_size=3, padding=1)
        self.conv14 = nn.Conv2d(256, 256, kernel_size=3, padding=1)

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv15 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.conv16 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        self.upconv4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv17 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.conv18 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.conv19 = nn.Conv2d(64, out_channels, kernel_size=1)

        self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)

    def test_step(self, batch, batch_idx):
        x, y = batch

        y_hat = self.forward(x)
        loss = F.cross_entropy(y_hat, y)

        self.accuracy(y_hat, y)

        self.log("test_accuracy", self.accuracy)
        self.log("test_loss", loss)

    def validation_step(self, batch, batch_nb):
        x, y = batch
        y = y.long()
        y_hat = self.forward(x)
        loss = nn.functional.cross_entropy(y_hat, y)
        self.validation_step_outputs.append(loss)
        self.log("val_loss", loss)
        return {'val_loss': loss}


    def forward(self, x):
       #Encoder
        x1 = F.relu(self.conv2(F.relu(self.conv1(x))))
        x2 = F.relu(self.conv4(F.relu(self.conv3(self.pool1(x1)))))
        x3 = F.relu(self.conv6(F.relu(self.conv5(self.pool2(x2)))))
        x4 = F.relu(self.conv8(F.relu(self.conv7(self.pool3(x3)))))
        x5 = F.relu(self.conv10(F.relu(self.conv9(self.pool4(x4)))))

        #Decoder
        x = F.relu(self.conv12(F.relu(self.conv11(torch.cat([x4, self.upconv1(x5)], 1)))))
        x = F.relu(self.conv14(F.relu(self.conv13(torch.cat([x3, self.upconv2(x)], 1)))))
        x = F.relu(self.conv16(F.relu(self.conv15(torch.cat([x2, self.upconv3(x)], 1)))))
        x = F.relu(self.conv18(F.relu(self.conv17(torch.cat([x1, self.upconv4(x)], 1)))))
        x = self.conv19(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        # x = torch.tensor(x, dtype=torch.float32)
        y = y.long()
        y_hat = self.forward(x)
        loss = nn.functional.cross_entropy(y_hat, y)
        iou = calculate_iou(y_hat,y)
        self.log('train_loss', loss)
        self.log('iou', iou)
        tensorboard_logs = {'train_loss': loss}
        return {'loss': loss, 'log': tensorboard_logs}


    def configure_optimizers(self):
        return optim.SGD(self.parameters(),lr=0.1,momentum=0.9,weight_decay=0.0005)

# U2Net Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class res_conv(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, dirate=1):
        super(res_conv, self).__init__()

        self.conv_s1 = nn.Conv2d(
            in_ch, out_ch, 3, padding=1 * dirate, dilation=1 * dirate
        )
        self.bn_s1 = nn.BatchNorm2d(out_ch)
        self.relu_s1 = nn.ReLU(inplace=True)

    def forward(self, x):

        layer_result = x
        xout = self.relu_s1(self.bn_s1(self.conv_s1(layer_result)))

        return xout



def _upsample_like(src, tar):

    src = F.upsample(src, size=tar.shape[2:], mode="bilinear")

    return src

class residual_block4F(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(residual_block4F, self).__init__()

        self.res_convin = res_conv(in_ch, out_ch, dirate=1)

        self.res_conv1 = res_conv(out_ch, mid_ch, dirate=1)
        self.res_conv2 = res_conv(mid_ch, mid_ch, dirate=2)
        self.res_conv3 = res_conv(mid_ch, mid_ch, dirate=4)

        self.res_conv4 = res_conv(mid_ch, mid_ch, dirate=8)

        self.res_conv3d = res_conv(mid_ch * 2, mid_ch, dirate=4)
        self.res_conv2d = res_conv(mid_ch * 2, mid_ch, dirate=2)
        self.res_conv1d = res_conv(mid_ch * 2, out_ch, dirate=1)

    def forward(self, x):

        layer_result = x

        layer_resultin = self.res_convin(layer_result)

        layer_result1 = self.res_conv1(layer_resultin)
        layer_result2 = self.res_conv2(layer_result1)
        layer_result3 = self.res_conv3(layer_result2)

        layer_result4 = self.res_conv4(layer_result3)

        layer_result3d = self.res_conv3d(torch.cat((layer_result4, layer_result3), 1))
        layer_result2d = self.res_conv2d(torch.cat((layer_result3d, layer_result2), 1))
        layer_result1d = self.res_conv1d(torch.cat((layer_result2d, layer_result1), 1))


        return layer_result1d + layer_resultin


class residual_block7(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(residual_block7, self).__init__()

        self.res_convin = res_conv(in_ch, out_ch, dirate=1)

        self.res_conv1 = res_conv(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv2 = res_conv(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv3 = res_conv(mid_ch, mid_ch, dirate=1)
        self.pool3 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv4 = res_conv(mid_ch, mid_ch, dirate=1)
        self.pool4 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv5 = res_conv(mid_ch, mid_ch, dirate=1)
        self.pool5 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv6 = res_conv(mid_ch, mid_ch, dirate=1)

        self.res_conv7 = res_conv(mid_ch, mid_ch, dirate=2)

        self.res_conv6d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv5d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv4d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv3d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv2d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv1d = res_conv(mid_ch * 2, out_ch, dirate=1)

    def forward(self, x):

        layer_result = x
        layer_resultin = self.res_convin(layer_result)

        layer_result1 = self.res_conv1(layer_resultin)
        layer_result = self.pool1(layer_result1)

        layer_result2 = self.res_conv2(layer_result)
        layer_result = self.pool2(layer_result2)

        layer_result3 = self.res_conv3(layer_result)
        layer_result = self.pool3(layer_result3)

        layer_result4 = self.res_conv4(layer_result)
        layer_result = self.pool4(layer_result4)

        layer_result5 = self.res_conv5(layer_result)
        layer_result = self.pool5(layer_result5)

        layer_result6 = self.res_conv6(layer_result)

        layer_result7 = self.res_conv7(layer_result6)

        layer_result6d = self.res_conv6d(torch.cat((layer_result7, layer_result6), 1))
        layer_result6dup = _upsample_like(layer_result6d, layer_result5)

        layer_result5d = self.res_conv5d(torch.cat((layer_result6dup, layer_result5), 1))
        layer_result5dup = _upsample_like(layer_result5d, layer_result4)

        layer_result4d = self.res_conv4d(torch.cat((layer_result5dup, layer_result4), 1))
        layer_result4dup = _upsample_like(layer_result4d, layer_result3)

        layer_result3d = self.res_conv3d(torch.cat((layer_result4dup, layer_result3), 1))
        layer_result3dup = _upsample_like(layer_result3d, layer_result2)

        layer_result2d = self.res_conv2d(torch.cat((layer_result3dup, layer_result2), 1))
        layer_result2dup = _upsample_like(layer_result2d, layer_result1)

        layer_result1d = self.res_conv1d(torch.cat((layer_result2dup, layer_result1), 1))



        return layer_result1d + layer_resultin






class residual_block6(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(residual_block6, self).__init__()

        self.res_convin = res_conv(in_ch, out_ch, dirate=1)

        self.res_conv1 = res_conv(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv2 = res_conv(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv3 = res_conv(mid_ch, mid_ch, dirate=1)
        self.pool3 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv4 = res_conv(mid_ch, mid_ch, dirate=1)
        self.pool4 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv5 = res_conv(mid_ch, mid_ch, dirate=1)

        self.res_conv6 = res_conv(mid_ch, mid_ch, dirate=2)

        self.res_conv5d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv4d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv3d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv2d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv1d = res_conv(mid_ch * 2, out_ch, dirate=1)

    def forward(self, x):

        layer_result = x

        layer_resultin = self.res_convin(layer_result)

        layer_result1 = self.res_conv1(layer_resultin)
        layer_result = self.pool1(layer_result1)

        layer_result2 = self.res_conv2(layer_result)
        layer_result = self.pool2(layer_result2)

        layer_result3 = self.res_conv3(layer_result)
        layer_result = self.pool3(layer_result3)

        layer_result4 = self.res_conv4(layer_result)
        layer_result = self.pool4(layer_result4)

        layer_result5 = self.res_conv5(layer_result)

        layer_result6 = self.res_conv6(layer_result5)

        layer_result5d = self.res_conv5d(torch.cat((layer_result6, layer_result5), 1))
        layer_result5dup = _upsample_like(layer_result5d, layer_result4)

        layer_result4d = self.res_conv4d(torch.cat((layer_result5dup, layer_result4), 1))
        layer_result4dup = _upsample_like(layer_result4d, layer_result3)

        layer_result3d = self.res_conv3d(torch.cat((layer_result4dup, layer_result3), 1))
        layer_result3dup = _upsample_like(layer_result3d, layer_result2)

        layer_result2d = self.res_conv2d(torch.cat((layer_result3dup, layer_result2), 1))
        layer_result2dup = _upsample_like(layer_result2d, layer_result1)

        layer_result1d = self.res_conv1d(torch.cat((layer_result2dup, layer_result1), 1))



        return layer_result1d + layer_resultin


class residual_block5(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(residual_block5, self).__init__()

        self.res_convin = res_conv(in_ch, out_ch, dirate=1)

        self.res_conv1 = res_conv(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv2 = res_conv(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv3 = res_conv(mid_ch, mid_ch, dirate=1)
        self.pool3 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv4 = res_conv(mid_ch, mid_ch, dirate=1)

        self.res_conv5 = res_conv(mid_ch, mid_ch, dirate=2)

        self.res_conv4d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv3d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv2d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv1d = res_conv(mid_ch * 2, out_ch, dirate=1)

    def forward(self, x):

        layer_result = x

        layer_resultin = self.res_convin(layer_result)

        layer_result1 = self.res_conv1(layer_resultin)
        layer_result = self.pool1(layer_result1)

        layer_result2 = self.res_conv2(layer_result)
        layer_result = self.pool2(layer_result2)

        layer_result3 = self.res_conv3(layer_result)
        layer_result = self.pool3(layer_result3)

        layer_result4 = self.res_conv4(layer_result)

        layer_result5 = self.res_conv5(layer_result4)

        layer_result4d = self.res_conv4d(torch.cat((layer_result5, layer_result4), 1))
        layer_result4dup = _upsample_like(layer_result4d, layer_result3)

        layer_result3d = self.res_conv3d(torch.cat((layer_result4dup, layer_result3), 1))
        layer_result3dup = _upsample_like(layer_result3d, layer_result2)

        layer_result2d = self.res_conv2d(torch.cat((layer_result3dup, layer_result2), 1))
        layer_result2dup = _upsample_like(layer_result2d, layer_result1)

        layer_result1d = self.res_conv1d(torch.cat((layer_result2dup, layer_result1), 1))



        return layer_result1d + layer_resultin


class residual_block4(nn.Module):
    def __init__(self, in_ch=3, mid_ch=12, out_ch=3):
        super(residual_block4, self).__init__()

        self.res_convin = res_conv(in_ch, out_ch, dirate=1)

        self.res_conv1 = res_conv(out_ch, mid_ch, dirate=1)
        self.pool1 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv2 = res_conv(mid_ch, mid_ch, dirate=1)
        self.pool2 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.res_conv3 = res_conv(mid_ch, mid_ch, dirate=1)

        self.res_conv4 = res_conv(mid_ch, mid_ch, dirate=2)

        self.res_conv3d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv2d = res_conv(mid_ch * 2, mid_ch, dirate=1)
        self.res_conv1d = res_conv(mid_ch * 2, out_ch, dirate=1)

    def forward(self, x):

        layer_result = x

        layer_resultin = self.res_convin(layer_result)

        layer_result1 = self.res_conv1(layer_resultin)
        layer_result = self.pool1(layer_result1)

        layer_result2 = self.res_conv2(layer_result)
        layer_result = self.pool2(layer_result2)

        layer_result3 = self.res_conv3(layer_result)

        layer_result4 = self.res_conv4(layer_result3)

        layer_result3d = self.res_conv3d(torch.cat((layer_result4, layer_result3), 1))
        layer_result3dup = _upsample_like(layer_result3d, layer_result2)

        layer_result2d = self.res_conv2d(torch.cat((layer_result3dup, layer_result2), 1))
        layer_result2dup = _upsample_like(layer_result2d, layer_result1)

        layer_result1d = self.res_conv1d(torch.cat((layer_result2dup, layer_result1), 1))



        return layer_result1d + layer_resultin





class U2NET(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super(U2NET, self).__init__()

        self.intm_result_output1 = residual_block7(in_ch, 32, 64)
        self.pool12 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.intm_result_output2 = residual_block6(64, 32, 128)
        self.pool23 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.intm_result_output3 = residual_block5(128, 64, 256)
        self.pool34 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.intm_result_output4 = residual_block4(256, 128, 512)
        self.pool45 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.intm_result_output5 = residual_block4F(512, 256, 512)
        self.pool56 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.intm_result_output6 = residual_block4F(512, 256, 512)

        # decoder
        self.intm_result_output5d = residual_block4F(1024, 256, 512)
        self.intm_result_output4d = residual_block4(1024, 128, 256)
        self.intm_result_output3d = residual_block5(512, 64, 128)
        self.intm_result_output2d = residual_block6(256, 32, 64)
        self.intm_result_output1d = residual_block7(128, 16, 64)

        self.intm_result1 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.intm_result2 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.intm_result3 = nn.Conv2d(128, out_ch, 3, padding=1)
        self.intm_result4 = nn.Conv2d(256, out_ch, 3, padding=1)
        self.intm_result5 = nn.Conv2d(512, out_ch, 3, padding=1)
        self.intm_result6 = nn.Conv2d(512, out_ch, 3, padding=1)

        self.outconv = nn.Conv2d(6 * out_ch, out_ch, 1)

    def forward(self, x):

        layer_result = x

        # intm_result_output 1
        layer_result1 = self.intm_result_output1(layer_result)
        layer_result = self.pool12(layer_result1)

        # intm_result_output 2
        layer_result2 = self.intm_result_output2(layer_result)
        layer_result = self.pool23(layer_result2)

        # intm_result_output 3
        layer_result3 = self.intm_result_output3(layer_result)
        layer_result = self.pool34(layer_result3)

        # intm_result_output 4
        layer_result4 = self.intm_result_output4(layer_result)
        layer_result = self.pool45(layer_result4)

        # intm_result_output 5
        layer_result5 = self.intm_result_output5(layer_result)
        layer_result = self.pool56(layer_result5)

        # intm_result_output 6
        layer_result6 = self.intm_result_output6(layer_result)
        layer_result6up = _upsample_like(layer_result6, layer_result5)

        # -------------------- decoder --------------------
        layer_result5d = self.intm_result_output5d(torch.cat((layer_result6up, layer_result5), 1))
        layer_result5dup = _upsample_like(layer_result5d, layer_result4)

        layer_result4d = self.intm_result_output4d(torch.cat((layer_result5dup, layer_result4), 1))
        layer_result4dup = _upsample_like(layer_result4d, layer_result3)

        layer_result3d = self.intm_result_output3d(torch.cat((layer_result4dup, layer_result3), 1))
        layer_result3dup = _upsample_like(layer_result3d, layer_result2)

        layer_result2d = self.intm_result_output2d(torch.cat((layer_result3dup, layer_result2), 1))
        layer_result2dup = _upsample_like(layer_result2d, layer_result1)

        layer_result1d = self.intm_result_output1d(torch.cat((layer_result2dup, layer_result1), 1))

        # intm_result output
        d1 = self.intm_result1(layer_result1d)

        d2 = self.intm_result2(layer_result2d)
        d2 = _upsample_like(d2, d1)

        d3 = self.intm_result3(layer_result3d)
        d3 = _upsample_like(d3, d1)

        d4 = self.intm_result4(layer_result4d)
        d4 = _upsample_like(d4, d1)

        d5 = self.intm_result5(layer_result5d)
        d5 = _upsample_like(d5, d1)

        d6 = self.intm_result6(layer_result6)
        d6 = _upsample_like(d6, d1)

        d0 = self.outconv(torch.cat((d1, d2, d3, d4, d5, d6), 1))



        return d0, d1, d2, d3, d4, d5, d6



class U2NETP(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super(U2NETP, self).__init__()

        self.intm_result_output1 = residual_block7(in_ch, 16, 64)
        self.pool12 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.intm_result_output2 = residual_block6(64, 16, 64)
        self.pool23 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.intm_result_output3 = residual_block5(64, 16, 64)
        self.pool34 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.intm_result_output4 = residual_block4(64, 16, 64)
        self.pool45 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.intm_result_output5 = residual_block4F(64, 16, 64)
        self.pool56 = nn.MaxPool2d(2, stride=2, ceil_mode=True)

        self.intm_result_output6 = residual_block4F(64, 16, 64)

        # decoder
        self.intm_result_output5d = residual_block4F(128, 16, 64)
        self.intm_result_output4d = residual_block4(128, 16, 64)
        self.intm_result_output3d = residual_block5(128, 16, 64)
        self.intm_result_output2d = residual_block6(128, 16, 64)
        self.intm_result_output1d = residual_block7(128, 16, 64)

        self.intm_result1 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.intm_result2 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.intm_result3 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.intm_result4 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.intm_result5 = nn.Conv2d(64, out_ch, 3, padding=1)
        self.intm_result6 = nn.Conv2d(64, out_ch, 3, padding=1)

        self.outconv = nn.Conv2d(6 * out_ch, out_ch, 1)

    def forward(self, x):

        layer_result = x

        # intm_result_output 1
        layer_result1 = self.intm_result_output1(layer_result)
        layer_result = self.pool12(layer_result1)

        # intm_result_output 2
        layer_result2 = self.intm_result_output2(layer_result)
        layer_result = self.pool23(layer_result2)

        # intm_result_output 3
        layer_result3 = self.intm_result_output3(layer_result)
        layer_result = self.pool34(layer_result3)

        # intm_result_output 4
        layer_result4 = self.intm_result_output4(layer_result)
        layer_result = self.pool45(layer_result4)

        # intm_result_output 5
        layer_result5 = self.intm_result_output5(layer_result)
        layer_result = self.pool56(layer_result5)

        # intm_result_output 6
        layer_result6 = self.intm_result_output6(layer_result)
        layer_result6up = _upsample_like(layer_result6, layer_result5)

        # decoder
        layer_result5d = self.intm_result_output5d(torch.cat((layer_result6up, layer_result5), 1))
        layer_result5dup = _upsample_like(layer_result5d, layer_result4)

        layer_result4d = self.intm_result_output4d(torch.cat((layer_result5dup, layer_result4), 1))
        layer_result4dup = _upsample_like(layer_result4d, layer_result3)

        layer_result3d = self.intm_result_output3d(torch.cat((layer_result4dup, layer_result3), 1))
        layer_result3dup = _upsample_like(layer_result3d, layer_result2)

        layer_result2d = self.intm_result_output2d(torch.cat((layer_result3dup, layer_result2), 1))
        layer_result2dup = _upsample_like(layer_result2d, layer_result1)

        layer_result1d = self.intm_result_output1d(torch.cat((layer_result2dup, layer_result1), 1))

        # intm_result output
        d1 = self.intm_result1(layer_result1d)

        d2 = self.intm_result2(layer_result2d)
        d2 = _upsample_like(d2, d1)

        d3 = self.intm_result3(layer_result3d)
        d3 = _upsample_like(d3, d1)

        d4 = self.intm_result4(layer_result4d)
        d4 = _upsample_like(d4, d1)

        d5 = self.intm_result5(layer_result5d)
        d5 = _upsample_like(d5, d1)

        d6 = self.intm_result6(layer_result6)
        d6 = _upsample_like(d6, d1)

        d0 = self.outconv(torch.cat((d1, d2, d3, d4, d5, d6), 1))



        return d0, d1, d2, d3, d4, d5, d6

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
import pytorch_lightning as pl

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchmetrics
import pandas as pd
from torchmetrics.functional import dice


def calculate_iou(prediction, target):
    value = dice(prediction, target)
    return value

#U2-net MODEL ARCHITECTURE
class U2net_pl(pl.LightningModule):
    def __init__(self,in_channels, out_channels):
        self.validation_step_outputs = []
        super(U2net_pl, self).__init__()
        self.model = U2NETP()
        self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)

    def test_step(self, batch, batch_idx):
        x, y = batch

        y_hat = self.forward(x)
        loss = F.cross_entropy(y_hat, y)

        self.accuracy(y_hat, y)

        self.log("test_accuracy", self.accuracy)
        self.log("test_loss", loss)

    def validation_step(self, batch, batch_nb):
        x, y = batch
        y = y.long()
        y_hat = self.forward(x)
        loss = nn.functional.cross_entropy(y_hat, y)
        self.validation_step_outputs.append(loss)
        self.log("val_loss", loss)
        return {'val_loss': loss}


    def forward(self, x):

        x = self.model(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y = y.long()
        y_hat = self.forward(x)
        loss = nn.functional.cross_entropy(y_hat, y)
        iou = calculate_iou(y_hat,y)
        self.log('train_loss', loss)
        self.log('iou', iou)
        tensorboard_logs = {'train_loss': loss}
        return {'loss': loss, 'log': tensorboard_logs}


    def configure_optimizers(self):
        return optim.Adam(self.parameters(),lr=0.01,momentum=0.9,weight_decay=0.0005)

# Training

In [ ]:
dataset_size = 35000
train_size = int(dataset_size * 0.85)
val_size = dataset_size - train_size
train_split, val_split = random_split(train_dataset,[train_size,val_size])
train_loader = torch.utils.data.DataLoader(train_split, batch_size=8, num_workers=1)
val_loader = torch.utils.data.DataLoader(val_split, batch_size=8, num_workers=1)

In [ ]:
early_stopping_callback = pl.callbacks.EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5
)

In [ ]:
from lightning.pytorch.loggers import TensorBoardLogger

logger = TensorBoardLogger("tb_logs", name="my_model")
trainer = pl.Trainer(accelerator="gpu", callbacks=[early_stopping_callback],max_epochs=100, check_val_every_n_epoch=1, logger=logger)

In [ ]:
model = Unet(3,47)
trainer.fit(model, train_loader, val_loader)

In [ ]:
model = U2net_pl(3,47)
trainer.fit(model, train_loader, val_loader)

Results

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=/content/lightning_logs